## Colab on CUDA programming

This Jupyter nodebook is to demonstrate how to use Colab to do CUDA programming. 

1. Open your Google Drive on Chrome. Upload the provided tensor_cuda folder to your Google drive. It contains a Jupyter notebook cuda_colab.ipynb
2. Open cuda_colab.ipynb by double clicking it. 
3. From Runtime -> Change runtime type, select T4 GPU, save
4. Run the following commands to ensure the output is like provided. 

In [ ]:
!echo "Hello from bash"
!cat /etc/os-release
!uname -a
!pwd
!ls -l
!lscpu
!nvidia-smi

Note that colab notebook runs on Ubuntu server, and binds with a session. The session is closed when notebook is closed. 
!command is run a Linux command with Jypyter's Python cell. 
!pwd shows the path of current directory of the jupyter nodebook, e.g. /content, binding with current session. A new content directory will be created everytime a notebook is opened, removed after the session is closed.  
!ls -l lists files and folders under the current working directory
!nvidia-smi shows information of NVIDIA devices

In [ ]:
!nvcc --version

The following cell writes a program to a file in Python cell. The file is written under the current working directory content. The file will be removed after the notebook is closed.  

In [ ]:
%%writefile vector_add.cu

#include <stdio.h>
#include <cuda_runtime.h>

__global__ void vectorAdd(int *a, int *b, int *c, int n) {
    int i = threadIdx.x + blockDim.x * blockIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}

int main() {
    int n = 100;
    size_t size = n * sizeof(int);

    // Allocate host memory
    int *h_a = (int *)malloc(size);
    int *h_b = (int *)malloc(size);
    int *h_c = (int *)malloc(size);

    // Initialize vectors
    for (int i = 0; i < n; i++) {
        h_a[i] = i;
        h_b[i] = 2 * i;
    }

    // Allocate device memory
    int *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, size);
    cudaMalloc(&d_b, size);
    cudaMalloc(&d_c, size);

    // Copy vectors from host to device
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    // Launch kernel
    int threadsPerBlock = 256;
    int blocksPerGrid = (n + threadsPerBlock - 1) / threadsPerBlock;
    vectorAdd<<<blocksPerGrid, threadsPerBlock>>>(d_a, d_b, d_c, n);

    // Copy result back to host
    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    // Print first 10 results
    for (int i = 0; i < 10; i++) {
        printf("%d + %d = %d\n", h_a[i], h_b[i], h_c[i]);
    }

    // Free memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}

Writing vector_add.cu


In [ ]:
!nvcc -arch=sm_75 vector_add.cu -o vector_add

In [ ]:
!./vector_add

0 + 0 = 0
1 + 2 = 3
2 + 4 = 6
3 + 6 = 9
4 + 8 = 12
5 + 10 = 15
6 + 12 = 18
7 + 14 = 21
8 + 16 = 24
9 + 18 = 27


You can run the program using nvprof to see the source usage.

In [ ]:
!nvprof ./vector_add

Save the stdout to a file

In [ ]:
!./vector_add > result.txt

In [ ]:
!ls

The following download the result.txt file to local machine.

In [ ]:
from google.colab import files
files.download('result.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

You can mount Google drive to the notebook's current session file system. There are thres ways:

1. By cell command: !drive.mount('/content/drive')
2. Click the file folder icon, then click the Google drive icon on the top.   
3. By the following Python code 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

After mouting, you copy files to Google's folder like the following.

In [ ]:
!cp result.txt /content/drive/MyDrive/tensor_cuda/result.txt

You can change session current working directory to Google Drive directory.

In [ ]:
import os

# Print the current working directory
print("Current directory:", os.getcwd())

# Create a new directory
new_dir = "drive/MyDrive/tensor_cuda"
# os.makedirs(new_dir, exist_ok=True)
# print(f"Created directory: {new_dir}")

# Change the working directory
os.chdir(new_dir)

# Print the new working directory
print("New current directory:", os.getcwd())

In [ ]:
!pwd
!ls -l

Then writing file and running program within the notebook will be in your Google's drive directory. 

In [ ]:
%%writefile dot_product.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void dotProduct(const float* A, const float* B, float* C, int N) {
    __shared__ float cache[1024];
    int i = threadIdx.x + blockDim.x * blockIdx.x;

    if(i < N) {
        cache[threadIdx.x] = A[i] * B[i];
    }

    __syncthreads();

    int i_half = blockDim.x / 2;
    while(i_half > 0) {
        if(threadIdx.x < i_half) {
            cache[threadIdx.x] += cache[threadIdx.x + i_half];
        }
        __syncthreads();
        i_half /= 2;
    }

    if(threadIdx.x == 0) {
        C[blockIdx.x] = cache[0];
    }
}

int main() {
    const int N = 1024;
    const int size = N * sizeof(float);

    float h_A[N], h_B[N];
    for (int i = 0; i < N; ++i) {
        h_A[i] = i;
        h_B[i] = i;
    }

    float *d_A, *d_B, *d_C;
    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, sizeof(float));

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dotProduct<<<1, N>>>(d_A, d_B, d_C, N);

    float result;
    cudaMemcpy(&result, d_C, sizeof(float), cudaMemcpyDeviceToHost);

    printf("Dot Product: %.1f\n", result);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing dot_product.cu


In [ ]:
!nvcc -arch=sm_75 dot_product.cu -o dot_product

In [ ]:
!./dot_product > dot_product_result.txt

It is more convenient to use terminal command to write, build and run programs. Colab provide terminal to the Ubuntu system. You open the terminal by clicking the terminal icon at the bottom-left. Then cd to the google drive directory tensor_cuda, and use commmand to build and run the required cpp and cu program. 

Once you get the CUDA environment working, next you can focus on writing required code of tensor programs. You can do it in three ways:
1. Write all source code in VS code, and then upload to Google Drive and build test there. Copilot can be used to help the development. 
2. Write the code in jupyter notebook and test it within the notebook. Gemini can be used to help the development.
3. Write the code in old style using simple text editor vi under the termial.  

In [ ]:
%%writefile mandelbrot.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

__global__ void mandelbrot(float* output, int width, int height, int maxIter) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= width || y >= height) return;

    float real = (x - width / 2.0f) * 4.0f / width;
    float imag = (y - height / 2.0f) * 4.0f / width;

    float zr = 0.0f, zi = 0.0f;
    int iter = 0;

    while (zr * zr + zi * zi < 4.0f && iter < maxIter) {
        float tmp = zr * zr - zi * zi + real;
        zi = 2.0f * zr * zi + imag;
        zr = tmp;
        iter++;
    }

    output[y * width + x] = (float)iter / maxIter;
}

int main() {
    const int width = 1024;
    const int height = 1024;
    const int maxIter = 1000;
    const size_t size = width * height * sizeof(float);

    // Allocate host memory
    float *h_output = (float *)malloc(size);

    // Allocate device memory
    float *d_output;
    cudaMalloc(&d_output, size);

    // Launch kernel
    dim3 block(16, 16);
    dim3 grid((width + 15) / 16, (height + 15) / 16);
    mandelbrot<<<grid, block>>>(d_output, width, height, maxIter);

    // Check for errors
    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        printf("CUDA error: %s\n", cudaGetErrorString(err));
        return 1;
    }

    // Wait for kernel to complete
    cudaDeviceSynchronize();

    // Copy result back to host
    cudaMemcpy(h_output, d_output, size, cudaMemcpyDeviceToHost);

    // Write to binary file for Python to read
    FILE *fp = fopen("mandelbrot_data.bin", "wb");
    if (fp) {
        fwrite(&width, sizeof(int), 1, fp);
        fwrite(&height, sizeof(int), 1, fp);
        fwrite(h_output, sizeof(float), width * height, fp);
        fclose(fp);
        printf("Mandelbrot data written to mandelbrot_data.bin\n");
    }

    // Free memory
    cudaFree(d_output);
    free(h_output);

    return 0;
}


Writing mandelbrot.cu


PermissionError: [Errno 13] Permission denied: 'mandelbrot.cu'

## GPU Graphics Generation and Visualization

The following example demonstrates generating graphics data (Mandelbrot set) on the GPU and visualizing it in Python using matplotlib.


In [ ]:
!nvcc -arch=sm_75 mandelbrot.cu -o mandelbrot


In [ ]:
!./mandelbrot


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import struct

# Read the binary data file generated by CUDA
with open('mandelbrot_data.bin', 'rb') as f:
    width = struct.unpack('i', f.read(4))[0]
    height = struct.unpack('i', f.read(4))[0]
    data = np.frombuffer(f.read(width * height * 4), dtype=np.float32)
    data = data.reshape((height, width))

# Create visualization
plt.figure(figsize=(12, 12))
plt.imshow(data, cmap='hot', origin='lower', interpolation='bilinear')
plt.colorbar(label='Iteration Count (normalized)')
plt.title('Mandelbrot Set - Generated on GPU', fontsize=16, fontweight='bold')
plt.xlabel('X', fontsize=12)
plt.ylabel('Y', fontsize=12)
plt.tight_layout()
plt.show()

print(f"Visualized Mandelbrot set: {width}x{height} pixels")


### Interactive Visualization with Different Color Maps

Let's create multiple visualizations with different color schemes:


In [ ]:
# Create a figure with multiple subplots showing different color maps
fig, axes = plt.subplots(2, 2, figsize=(16, 16))

color_maps = ['hot', 'viridis', 'plasma', 'inferno']
titles = ['Hot Colormap', 'Viridis Colormap', 'Plasma Colormap', 'Inferno Colormap']

for idx, (ax, cmap, title) in enumerate(zip(axes.flat, color_maps, titles)):
    im = ax.imshow(data, cmap=cmap, origin='lower', interpolation='bilinear')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('X', fontsize=10)
    ax.set_ylabel('Y', fontsize=10)
    plt.colorbar(im, ax=ax, label='Normalized Iterations')

plt.suptitle('Mandelbrot Set Visualizations - GPU Generated', 
             fontsize=18, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()


### Generate Julia Set on GPU

Let's create another fractal - the Julia set - and visualize it:


In [ ]:
%%writefile julia_set.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

__global__ void juliaSet(float* output, int width, int height, int maxIter, 
                         float c_real, float c_imag) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= width || y >= height) return;

    float real = (x - width / 2.0f) * 4.0f / width;
    float imag = (y - height / 2.0f) * 4.0f / width;

    int iter = 0;
    while (real * real + imag * imag < 4.0f && iter < maxIter) {
        float tmp = real * real - imag * imag + c_real;
        imag = 2.0f * real * imag + c_imag;
        real = tmp;
        iter++;
    }

    output[y * width + x] = (float)iter / maxIter;
}

int main() {
    const int width = 1024;
    const int height = 1024;
    const int maxIter = 1000;
    const float c_real = -0.7269f;  // Julia set parameter
    const float c_imag = 0.1889f;
    const size_t size = width * height * sizeof(float);

    float *h_output = (float *)malloc(size);
    float *d_output;
    cudaMalloc(&d_output, size);

    dim3 block(16, 16);
    dim3 grid((width + 15) / 16, (height + 15) / 16);
    juliaSet<<<grid, block>>>(d_output, width, height, maxIter, c_real, c_imag);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        printf("CUDA error: %s\n", cudaGetErrorString(err));
        return 1;
    }

    cudaDeviceSynchronize();
    cudaMemcpy(h_output, d_output, size, cudaMemcpyDeviceToHost);

    FILE *fp = fopen("julia_data.bin", "wb");
    if (fp) {
        fwrite(&width, sizeof(int), 1, fp);
        fwrite(&height, sizeof(int), 1, fp);
        fwrite(h_output, sizeof(float), width * height, fp);
        fclose(fp);
        printf("Julia set data written to julia_data.bin\n");
    }

    cudaFree(d_output);
    free(h_output);

    return 0;
}


In [ ]:
!nvcc -arch=sm_75 julia_set.cu -o julia_set && ./julia_set


In [ ]:
# Read Julia set data
with open('julia_data.bin', 'rb') as f:
    width_j = struct.unpack('i', f.read(4))[0]
    height_j = struct.unpack('i', f.read(4))[0]
    data_j = np.frombuffer(f.read(width_j * height_j * 4), dtype=np.float32)
    data_j = data_j.reshape((height_j, width_j))

# Create side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Mandelbrot set
im1 = axes[0].imshow(data, cmap='turbo', origin='lower', interpolation='bilinear')
axes[0].set_title('Mandelbrot Set (GPU Generated)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('X', fontsize=12)
axes[0].set_ylabel('Y', fontsize=12)
plt.colorbar(im1, ax=axes[0], label='Normalized Iterations')

# Julia set
im2 = axes[1].imshow(data_j, cmap='turbo', origin='lower', interpolation='bilinear')
axes[1].set_title('Julia Set (GPU Generated)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('X', fontsize=12)
axes[1].set_ylabel('Y', fontsize=12)
plt.colorbar(im2, ax=axes[1], label='Normalized Iterations')

plt.suptitle('Fractal Visualizations - GPU Accelerated', 
             fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()


### Performance Comparison: GPU vs CPU

Let's compare the performance of GPU vs CPU for generating the Mandelbrot set:


In [ ]:
import time

# CPU implementation for comparison
def mandelbrot_cpu(width, height, max_iter=1000):
    output = np.zeros((height, width), dtype=np.float32)
    for y in range(height):
        for x in range(width):
            real = (x - width / 2.0) * 4.0 / width
            imag = (y - height / 2.0) * 4.0 / width
            
            zr, zi = 0.0, 0.0
            iter_count = 0
            
            while zr * zr + zi * zi < 4.0 and iter_count < max_iter:
                tmp = zr * zr - zi * zi + real
                zi = 2.0 * zr * zi + imag
                zr = tmp
                iter_count += 1
            
            output[y, x] = iter_count / max_iter
    return output

# Test with smaller size for CPU (since it's slower)
test_size = 512
print(f"Testing with {test_size}x{test_size} pixels...")

# CPU timing
start = time.time()
cpu_result = mandelbrot_cpu(test_size, test_size)
cpu_time = time.time() - start
print(f"CPU time: {cpu_time:.4f} seconds")


In [ ]:
%%writefile mandelbrot_small.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

__global__ void mandelbrot(float* output, int width, int height, int maxIter) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= width || y >= height) return;
    float real = (x - width / 2.0f) * 4.0f / width;
    float imag = (y - height / 2.0f) * 4.0f / width;
    float zr = 0.0f, zi = 0.0f;
    int iter = 0;
    while (zr * zr + zi * zi < 4.0f && iter < maxIter) {
        float tmp = zr * zr - zi * zi + real;
        zi = 2.0f * zr * zi + imag;
        zr = tmp;
        iter++;
    }
    output[y * width + x] = (float)iter / maxIter;
}

int main(int argc, char* argv[]) {
    int width = (argc > 1) ? atoi(argv[1]) : 512;
    int height = (argc > 2) ? atoi(argv[2]) : 512;
    int maxIter = 1000;
    size_t size = width * height * sizeof(float);
    float *h_output = (float *)malloc(size);
    float *d_output;
    cudaMalloc(&d_output, size);
    dim3 block(16, 16);
    dim3 grid((width + 15) / 16, (height + 15) / 16);
    
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);
    
    mandelbrot<<<grid, block>>>(d_output, width, height, maxIter);
    cudaDeviceSynchronize();
    
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float gpu_time_ms = 0;
    cudaEventElapsedTime(&gpu_time_ms, start, stop);
    
    cudaMemcpy(h_output, d_output, size, cudaMemcpyDeviceToHost);
    printf("GPU time: %.4f seconds\n", gpu_time_ms / 1000.0f);
    
    cudaFree(d_output);
    free(h_output);
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 mandelbrot_small.cu -o mandelbrot_small
import subprocess
result = subprocess.run(['./mandelbrot_small', '512', '512'], 
                       capture_output=True, text=True)
print(result.stdout)
gpu_time = float(result.stdout.split('GPU time: ')[1].split(' seconds')[0])


In [ ]:
# Display performance comparison
print("\n" + "="*50)
print("PERFORMANCE COMPARISON")
print("="*50)
print(f"CPU Implementation: {cpu_time:.4f} seconds")
print(f"GPU Implementation: {gpu_time:.4f} seconds")
speedup = cpu_time / gpu_time
print(f"Speedup: {speedup:.1f}x faster on GPU")
print("="*50)
print("\nThe GPU provides massive parallelization, allowing")
print("thousands of threads to compute pixels simultaneously!")

# Visualize the comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].imshow(cpu_result, cmap='turbo', origin='lower', interpolation='bilinear')
axes[0].set_title(f'CPU Generated ({cpu_time:.4f}s)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('X', fontsize=12)
axes[0].set_ylabel('Y', fontsize=12)

# Read GPU result for comparison - use a simple crop to match size
with open('mandelbrot_data.bin', 'rb') as f:
    w = struct.unpack('i', f.read(4))[0]
    h = struct.unpack('i', f.read(4))[0]
    gpu_data = np.frombuffer(f.read(w * h * 4), dtype=np.float32)
    gpu_data = gpu_data.reshape((h, w))
    # Crop center portion to match CPU result size
    start_y = (h - test_size) // 2
    start_x = (w - test_size) // 2
    gpu_cropped = gpu_data[start_y:start_y+test_size, start_x:start_x+test_size]

axes[1].imshow(gpu_cropped, cmap='turbo', origin='lower', interpolation='bilinear')
axes[1].set_title(f'GPU Generated ({gpu_time:.4f}s)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('X', fontsize=12)
axes[1].set_ylabel('Y', fontsize=12)

plt.suptitle(f'CPU vs GPU Performance Comparison ({speedup:.1f}x speedup)', 
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()
